In [ ]:
import numpy as np

ket0 = np.array([1.0 + 0.0j, 0.0 + 0.0j])
ket1 = np.array([0.0 + 0.0j, 1.0 + 0.0j])


def kron_all(vectors):
    output = vectors[0]
    for vector in vectors[1:]:
        output = np.kron(output, vector)
    return output


def ry(theta: float) -> np.ndarray:
    return np.array(
        [
            [np.cos(theta / 2), -np.sin(theta / 2)],
            [np.sin(theta / 2),  np.cos(theta / 2)],
        ],
        dtype=complex,
    )


def basis_encode(bits: np.ndarray) -> np.ndarray:
    states = [ket1 if bit else ket0 for bit in bits]
    return kron_all(states)


def angle_encode(features: np.ndarray, scale: float = 1.0) -> np.ndarray:
    states = [ry(scale * value) @ ket0 for value in features]
    return kron_all(states)


def amplitude_encode(features: np.ndarray) -> np.ndarray:
    features = np.asarray(features, dtype=float)
    target_size = 1 << int(np.ceil(np.log2(len(features))))
    padded = np.zeros(target_size, dtype=complex)
    padded[: len(features)] = features
    norm = np.linalg.norm(padded)
    if norm == 0:
        raise ValueError("Amplitude encoding requires a nonzero vector.")
    return padded / norm


bits = np.array([1, 0, 1], dtype=int)
angles = np.array([0.2, -0.4, 0.7])
values = np.array([0.5, -1.0, 0.25, 0.75])

print("basis:", basis_encode(bits))
print("angle:", angle_encode(angles))
print("amplitude:", amplitude_encode(values))

In [ ]:
import numpy as np


def equal_up_to_global_phase(a: np.ndarray, b: np.ndarray, atol: float = 1e-8) -> bool:
    index = np.argmax(np.abs(b))
    phase = a[index] / b[index]
    return np.allclose(a, phase * b, atol=atol)


x = np.array([0.25, -0.4])
wrapped = x + 2 * np.pi

u = np.array([1.0, 2.0, -1.0, 0.5])
v = 10.0 * u

print("angle collision:", equal_up_to_global_phase(angle_encode(x), angle_encode(wrapped)))
print("amplitude collision:", np.allclose(amplitude_encode(u), amplitude_encode(v)))

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=200, noise=0.12, random_state=7)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=7,
    stratify=y,
)

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler


def trig_features(X: np.ndarray) -> np.ndarray:
    return np.concatenate([np.sin(X), np.cos(X)], axis=1)


baseline = Pipeline(
    [
        ("trig", FunctionTransformer(trig_features)),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000)),
    ]
)

baseline.fit(X_train, y_train)
print("test accuracy:", baseline.score(X_test, y_test))